# Практикум по логистической регрессии: группы 2 и 4

## Подготовка к работе

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from matplotlib import pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
job = pd.read_csv("https://raw.githubusercontent.com/allatambov/PyPerm25/refs/heads/main/HR.csv")
job.head()

In [ ]:
job["Department"].value_counts()

In [ ]:
df = job[job["Department"] == "IT"]
df.describe()

In [ ]:
df["salary"].value_counts()

## Часть 1: строим модель с `statsmodels` (статистический подход)

### Оценка модели и интерпретация коэффициентов

In [ ]:
equation = """
left ~ satisfaction_level + 
number_project + salary + time_spend_company
"""

In [ ]:
logit01 = smf.logit(equation, data = df).fit()
print(logit01.summary())

In [ ]:
print(logit01.params)

In [ ]:
names = logit01.params.index
coefs = logit01.params.values.round(2)
print(*zip(names, coefs), sep = "\n")

In [ ]:
print(np.exp(coefs))

In [ ]:
print(*zip(names, np.exp(coefs).round(2)), sep = "\n")

In [ ]:
marg_effect_mean = logit01.get_margeff(at = "mean", method = "dydx")
print(marg_effect_mean.summary())

In [ ]:
marg_effect_median = logit01.get_margeff(at = "median", method = "dydx")
print(marg_effect_median.summary())

In [ ]:
ave_marg_effect = logit01.get_margeff(at = "overall", method = "dydx")
print(ave_marg_effect.summary())

### Оценка качества модели: ROC-кривая и AUC

In [ ]:
df["prob"] =  logit01.predict()
df.head()

In [ ]:
fpr, tpr, thresholds = roc_curve(df["left"], df["prob"])

print("Cutoff points: ", thresholds[::50])
print("FPR: ", fpr[::50])
print("TPR: ", tpr[::50])

In [ ]:
logit_roc_auc = roc_auc_score(df["left"], df["prob"])
print("AUC:", round(logit_roc_auc, 2))

In [ ]:
plt.figure(dpi = 300)
plt.plot(fpr, tpr, label = f'Model 01 (AUC = {logit_roc_auc:.2f})' )
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC curve')
plt.legend(loc = "lower right")
plt.show()

### Предсказание вероятностей

In [ ]:
logit01.predict(exog = {"salary" : "low", 
                       "satisfaction_level" : 0.2, 
                       "number_project" : 3, 
                        "time_spend_company" : 2})

In [ ]:
logit01.predict(exog = {"salary" : ["low", "high", "medium"], 
                       "satisfaction_level" : [0.2, 0.8, 0.5],
                       "number_project" : [3, 1, 5], 
                       "time_spend_company" : [2, 5, 1]})

### Классификация с помощью модели

In [ ]:
tab = logit01.pred_table()
print(tab)

In [ ]:
TN = tab[0, 0]
TP = tab[1, 1]
FN = tab[1, 0]
FP = tab[0, 1]

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)

print("Sensitivity:", round(sensitivity, 2)) 
print("Specificity:", round(specificity, 2))

In [ ]:
df["left"].value_counts(normalize = True)

In [ ]:
tab2 = logit01.pred_table(threshold = 0.222)

TN2 = tab2[0, 0]
TP2 = tab2[1, 1]
FN2 = tab2[1, 0]
FP2 = tab2[0, 1]

sensitivity2 = TP2 / (TP2 + FN2)
specificity2 = TN2 / (TN2 + FP2)

print("Sensitivity:", round(sensitivity2, 2)) 
print("Specificity:", round(specificity2, 2))

In [ ]:
plt.figure(dpi = 300)
plt.plot(thresholds, tpr, label = "Sensitivity")
plt.plot(thresholds, 1 - fpr, label = "Specificity")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Cutoff point')
plt.ylabel('Rates')
plt.legend()
plt.show()

## Часть 2: строим модель с `sklearn` (машинное обучение)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
salary = pd.get_dummies(df["salary"])
print(salary.head())

In [ ]:
fin = pd.concat([df, salary], axis = 1)
fin.head()

In [ ]:
y = fin["left"]
X = fin[["satisfaction_level", "number_project", 
         "low", "medium", "time_spend_company"]]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.15, 
                                                    random_state=480)

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)

In [ ]:
print("Intercept:", clf.intercept_)
print("Coeffs:", clf.coef_)

In [ ]:
accuracy = accuracy_score(y_test, clf.predict(X_test))
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
clf.predict(X_test)

In [ ]:
clf.predict_proba(X_test)

In [ ]:
clf.predict_proba(X_test)[:, 1]

In [ ]:
auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
print("AUC:", round(auc, 2))

In [ ]:
clf.predict_proba(X_test)[:, 1] >= 0.5

### Задание

Вычислите значение *accuracy* для модели, изменив пороговое значение вероятности на долю 1 в тестовой выборке, округленную до сотых.